# Industrial IoT Predictive Maintenance & Multi-Fault Classification Pipeline

This project builds a production-ready predictive maintenance model handling severe class imbalance to predict specific machine failure modes (Heat Dissipation, Power Failure, Overstrain, Tool Wear).

## 1. Import libraries

In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
warnings.filterwarnings('ignore')

# Advanced Modeling & Imbalance Handling
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import RobustScaler, LabelEncoder
from imblearn.over_sampling import SMOTENC

# Explainability & Serialization
import shap
import joblib

## 2. Load the dataset

In [8]:
import kagglehub

# 1. Ingest Data
path = kagglehub.dataset_download("shivamb/machine-predictive-maintenance-classification")
df = pd.read_csv(os.path.join(path, "predictive_maintenance.csv"))

# 2. Automated Pipeline Validation (Fails fast if data is corrupted)
assert df.isnull().sum().sum() == 0, "Pipeline Error: Dataset contains missing values."
assert 'Target' in df.columns and 'Failure Type' in df.columns, "Pipeline Error: Target columns missing."

# 3. Immediate Sanitization
df = df.drop(columns=['UDI', 'Product ID'])

# 4. Imbalance Quantification
print(f"Sanitized Dataset Shape: {df.shape}\n")
print("Failure Type Distribution (Class Imbalance %):")
print(df['Failure Type'].value_counts(normalize=True) * 100)

Using Colab cache for faster access to the 'machine-predictive-maintenance-classification' dataset.
Sanitized Dataset Shape: (10000, 8)

Failure Type Distribution (Class Imbalance %):
Failure Type
No Failure                  96.52
Heat Dissipation Failure     1.12
Power Failure                0.95
Overstrain Failure           0.78
Tool Wear Failure            0.45
Random Failures              0.18
Name: proportion, dtype: float64


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Type                     10000 non-null  object 
 1   Air temperature [K]      10000 non-null  float64
 2   Process temperature [K]  10000 non-null  float64
 3   Rotational speed [rpm]   10000 non-null  int64  
 4   Torque [Nm]              10000 non-null  float64
 5   Tool wear [min]          10000 non-null  int64  
 6   Target                   10000 non-null  int64  
 7   Failure Type             10000 non-null  object 
dtypes: float64(3), int64(3), object(2)
memory usage: 625.1+ KB


In [10]:
df.describe()


,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Target
count,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,300.004930,310.005560,1538.776100,39.986910,107.951000,0.033900
std,2.000259,1.483734,179.284096,9.968934,63.654147,0.180981
min,295.300000,305.700000,1168.000000,3.800000,0.000000,0.000000
25%,298.300000,308.800000,1423.000000,33.200000,53.000000,0.000000
50%,300.100000,310.100000,1503.000000,40.100000,108.000000,0.000000
75%,301.500000,311.100000,1612.000000,46.800000,162.000000,0.000000
max,304.500000,313.800000,2886.000000,76.600000,253.000000,1.000000


In [11]:
df.shape

(10000, 8)

## 3. Feature engineering and target encoding

In [12]:
# 1. Separate Features (X) and Target (y)
X = df.drop(columns=['Target', 'Failure Type'])
y = df['Failure Type']

# 2. Encode the Multi-Class Target
target_encoder = LabelEncoder()
y_encoded = target_encoder.fit_transform(y)

# Save the encoder for the FastAPI backend
joblib.dump(target_encoder, 'target_encoder.joblib')

# 3. Identify categorical and numerical columns for SMOTENC and Scaling
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Get the indices of categorical columns for SMOTENC
categorical_features_indices = [X.columns.get_loc(col) for col in categorical_cols]

print(f"Categorical Features: {categorical_cols}")
print(f"Numerical Features: {numerical_cols}")
print(f"Target Classes Mapping: {dict(zip(target_encoder.classes_, target_encoder.transform(target_encoder.classes_)))}")

Categorical Features: ['Type']
Numerical Features: ['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']
Target Classes Mapping: {'Heat Dissipation Failure': np.int64(0), 'No Failure': np.int64(1), 'Overstrain Failure': np.int64(2), 'Power Failure': np.int64(3), 'Random Failures': np.int64(4), 'Tool Wear Failure': np.int64(5)}


## 4. Stratified split and imbalance handling (SMOTENC)

In [13]:
# 1. Stratified Train-Test Split (Crucial for imbalanced datasets)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"Original Training Shape: {X_train.shape}")

# 2. Apply SMOTENC to the Training Data ONLY
print("Applying SMOTENC (This may take a moment)...")
smote_nc = SMOTENC(categorical_features=categorical_features_indices, random_state=42)
X_train_resampled, y_train_resampled = smote_nc.fit_resample(X_train, y_train)

print(f"Resampled Training Shape: {X_train_resampled.shape}")
print("New Class Distribution in Training Data:")
print(pd.Series(y_train_resampled).value_counts(normalize=True) * 100)

Original Training Shape: (8000, 6)
Applying SMOTENC (This may take a moment)...
Resampled Training Shape: (46332, 6)
New Class Distribution in Training Data:
1    16.666667
5    16.666667
4    16.666667
3    16.666667
0    16.666667
2    16.666667
Name: proportion, dtype: float64


## 5. Model training, evaluation and export

In [17]:
import re
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# 1. Clean feature names (XGBoost does not allow [, ], or < in column names)
X_train_resampled.columns = [re.sub(r'[\[\]<]', '', col) for col in X_train_resampled.columns]
X_test.columns = [re.sub(r'[\[\]<]', '', col) for col in X_test.columns]

# 2. Convert 'object' columns to 'category' for XGBoost
X_train_resampled['Type'] = X_train_resampled['Type'].astype('category')
X_test['Type'] = X_test['Type'].astype('category')

# 3. Initialize the XGBoost Classifier
xgb_model = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=len(np.unique(y_encoded)),
    random_state=42,
    eval_metric='mlogloss',
    enable_categorical=True
)

# 4. Train the model on the BALANCED training data
print("Training XGBoost Model (This may take a moment)...")
xgb_model.fit(X_train_resampled, y_train_resampled)

# 5. Predict on the UNSEEN, UNBALANCED test data
print("Evaluating Model on Test Data...")
y_pred = xgb_model.predict(X_test)

# 6. Evaluation Metrics
print(f"\nOverall Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=target_encoder.classes_))

# 7. Save the trained model for the FastAPI backend
joblib.dump(xgb_model, 'xgboost_predictive_maintenance.joblib')
print("\nModel saved successfully as 'xgboost_predictive_maintenance.joblib'")

Training XGBoost Model (This may take a moment)...
Evaluating Model on Test Data...

Overall Accuracy: 0.9650

Classification Report:
                          precision    recall  f1-score   support

Heat Dissipation Failure       0.96      1.00      0.98        22
              No Failure       0.99      0.97      0.98      1930
      Overstrain Failure       0.76      1.00      0.86        16
           Power Failure       0.73      0.84      0.78        19
         Random Failures       0.00      0.00      0.00         4
       Tool Wear Failure       0.00      0.00      0.00         9

                accuracy                           0.96      2000
               macro avg       0.57      0.64      0.60      2000
            weighted avg       0.98      0.96      0.97      2000


Model saved successfully as 'xgboost_predictive_maintenance.joblib'
